# Analyze your sequences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/analyze_sequences.ipynb)

Load your IDRs, summarize their composition, extract embeddings, and find similar sequences.
Outputs: a sequence table, embedding arrays, nearest-neighbor table, and PCA plot.

Run cells from top to bottom. In Colab select **Runtime → Change runtime type → GPU**.
A GPU is recommended; CPU works but is slower. Runtime and peak memory depend on sequence
length, model, and hardware; timings are printed below rather than promising a fixed runtime.
First use downloads model weights. Outputs are written under `OUT_DIR`; rerunning replaces
files with the same names. Download that folder from Colab before ending the session.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git@v1"])

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record, read_fasta, parse_idr_header
print("Python:", sys.version.split()[0])
import idiom
print("IDiom:", idiom.__file__)


# Locate the companion helper in a clone, or download it for standalone Colab use.
helper_dir = next((p for p in (Path.cwd(), Path.cwd() / "cookbook/notebooks")
                   if (p / "workflow_utils.py").is_file()), None)
if helper_dir is None:
    from urllib.request import urlretrieve
    helper_dir = Path(".idiom_notebook_helpers")
    helper_dir.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/rotskoff-group/idiom/main/"
                "cookbook/notebooks/workflow_utils.py", helper_dir / "workflow_utils.py")
sys.path.insert(0, str(helper_dir.resolve()))
from workflow_utils import (AA, DEMO, load_inputs, idr_sequence, isolated, check_context,
                            summaries, write_fasta, save_run)


## Inputs and validation

Use `INPUT_MODE="idr"` for FASTA records that are **already isolated IDRs** (ordinary headers
are accepted). Use `INPUT_MODE="annotated"` for full proteins: the first header token must
end in `_IDR_x-y`, with **1-based inclusive** coordinates. This notebook does not predict IDR
boundaries. Python slices use 0-based, end-exclusive coordinates.

`INPUT_FASTA=None` uses six small illustrative sequences, not experimentally labeled examples.
Set a local path to analyze your own file (upload it using the Colab Files pane).
The audit table reports rejected records and records outside the sample limit. Empty sequences,
noncanonical residues, and invalid annotations are not silently repaired. Repeated accessions
remain distinct through `record_id`; duplicate IDR sequences are reported for your review.


In [ ]:
INPUT_FASTA = None
INPUT_MODE = "idr"
MAX_RECORDS = 32 # None uses all valid records; pairwise comparisons grow quadratically
MODEL = "jxliu2/idiom-20M" # Smaller model for a first run
LAYER = 5 # Zero-based; for idiom-300M use e.g. layer 18
DEVICE = "auto"
POOL = "mean" # "mean" averages IDR residues; "last" selects the final IDR residue
USE_FLANKS = False # True embeds annotated full proteins with their flanking context
OUT_DIR = Path("analysis_outputs")


In [ ]:
started = time.perf_counter()


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
records, audit = load_inputs(INPUT_FASTA, INPUT_MODE, MAX_RECORDS)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
display(audit)
if not records:
    raise ValueError("No accepted records. Review input_audit.csv.")
summary = summaries(records, audit)
summary.to_csv(OUT_DIR / "sequence_summary.csv", index=False)
print(f"Accepted {len(records)} records; {summary.sequence.duplicated().sum()} duplicate IDR sequences")
display(summary[["record_id", "accession", "length", "charged_fraction", "net_charge_per_residue"]])


## Embed the IDRs

By default, extract each IDR before embedding so inputs share the same context convention.
`USE_FLANKS=True` retains annotated protein flanks as context; only IDR representations are returned.
`POOL="mean"` averages IDR residue vectors; `POOL="last"` selects the final IDR residue, excluding EOS.
Compare embeddings made with the same model, layer, pooling, and context convention. The API processes
records individually; reduce `MAX_RECORDS` to shorten the run.


In [ ]:
model = IDiom.from_pretrained(MODEL, device=DEVICE)
if not 0 <= LAYER < model.model.cfg.n_layers:
    raise ValueError("LAYER is outside this model's transformer blocks.")
if POOL not in ("mean", "last"):
    raise ValueError('POOL must be "mean" or "last" for sequence-level comparisons.')
embedding_records = records if USE_FLANKS else isolated(records)
check_context(embedding_records, model.model.cfg.max_seq_len, include_flanks=USE_FLANKS)
values, index = model.embed(embedding_records, layers=[LAYER], pool=POOL)[LAYER]
np.save(OUT_DIR / "embeddings.npy", values)
embedding_index = summary[["record_id", "accession"]].copy()
embedding_index.insert(0, "embedding_row", np.arange(len(values)))
embedding_index.to_csv(OUT_DIR / "embedding_index.csv", index=False)
print(values.shape, "on", model.device)


## Find similar sequences in the original embedding space

Cosine similarity compares embedding directions. Self matches are excluded; exact duplicate
sequences are retained and labeled. Similarity is exploratory, not a functional annotation.
For very large inputs, use a nearest-neighbor index instead of this all-pairs matrix.


In [ ]:
normalized = values / np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-12)
similarity = normalized @ normalized.T
np.fill_diagonal(similarity, -np.inf)
neighbors = []
for i in range(len(records)):
    for j in np.argsort(-similarity[i], kind="stable")[:min(3, len(records) - 1)]:
        neighbors.append(dict(query=records[i].accession, neighbor=records[j].accession,
                              cosine_similarity=float(similarity[i, j]),
                              identical_idr=idr_sequence(records[i]) == idr_sequence(records[j])))
neighbors = pd.DataFrame(neighbors, columns=["query", "neighbor", "cosine_similarity", "identical_idr"])
neighbors.to_csv(OUT_DIR / "nearest_neighbors.csv", index=False)
display(neighbors)


## Visualize a projection

PCA is fitted to this input set using NumPy SVD. Distances in two dimensions can hide differences
present in the full embeddings; use the table above for similarity comparisons. Repeated or
identical inputs may yield fewer than two meaningful components.


In [ ]:
if len(values) >= 2 and np.any(values != values[0]):
    centered = values - values.mean(axis=0)
    u, singular, _ = np.linalg.svd(centered, full_matrices=False)
    xy = u[:, :2] * singular[:2]
    explained = singular[:2] ** 2 / np.sum(singular ** 2)
    fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
    points = ax.scatter(xy[:, 0], xy[:, 1], c=summary.length, cmap="viridis")
    if len(records) <= 20:
        for r, point in zip(records, xy):
            ax.annotate(r.accession, point, fontsize=8)
    ax.set(xlabel=f"PC1 ({explained[0]:.1%})", ylabel=f"PC2 ({explained[1]:.1%})")
    fig.colorbar(points, ax=ax, label="IDR length")
    fig.savefig(OUT_DIR / "embedding_pca.png", dpi=160)
    plt.show()
    pd.DataFrame({"record_id": summary.record_id, "PC1": xy[:, 0], "PC2": xy[:, 1]}).to_csv(
        OUT_DIR / "embedding_pca.csv", index=False)
else:
    print("PCA needs at least two nonidentical embeddings.")
save_run(OUT_DIR, dict(input=INPUT_FASTA, mode=INPUT_MODE, max_records=MAX_RECORDS,
                       model=MODEL, layer=LAYER, pool=POOL, use_flanks=USE_FLANKS, device=str(model.device)), elapsed=time.perf_counter() - started)
print(f"Elapsed including model load: {time.perf_counter() - started:.1f} s")


## Optional: residue embeddings for one sequence

Only IDR residue rows are returned, in original IDR sequence order, even when flanks provide context.
`source_pos` is the 0-based position in the record passed to `embed`; use it to join annotations.
When IDRs were extracted first, add the original IDR start to recover protein coordinates.


In [ ]:
residue_values, residue_index = model.embed(embedding_records[:1], layers=[LAYER], pool="none")[LAYER]
residue_table = pd.DataFrame(residue_index)
residue_table["protein_position_1based"] = residue_table.source_pos + 1 + (0 if USE_FLANKS else records[0].idr_start)
np.save(OUT_DIR / "first_sequence_residue_embeddings.npy", residue_values)
residue_table.to_csv(OUT_DIR / "first_sequence_residue_index.csv", index=False)
display(residue_table.head())


## Use the outputs

Keep `input_audit.csv` with your results: `record_id` connects exported rows to the original
accession and IDR span, even when accessions repeat. `run.json` records the settings and installed
package versions. Record exact model revisions separately when freezing a published analysis.

Next: [inspect SAE features](inspect_sae_features.ipynb) to localize activations, or
[compare sequence sets](compare_sequence_sets.ipynb) to evaluate a design set against references.
